# TML 1903 — A100 Layout FINAL
Fresh launcher. VPS is the controller; Colab only computes layout on CUDA and uploads results immediately.


In [ ]:
import os, shutil, subprocess, sys
REPO='/content/Tennis-OCR-Pipeline'; VENV='/content/tml-layout-py312'; VENV_PY=f'{VENV}/bin/python'
shutil.rmtree(REPO,ignore_errors=True); shutil.rmtree(VENV,ignore_errors=True)
subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
UV=shutil.which('uv'); assert UV, 'uv not found'
subprocess.run([UV,'python','install','3.12'],check=True)
subprocess.run([UV,'venv','--seed','--python','3.12',VENV],check=True)
PADDLE='https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl'
subprocess.run([VENV_PY,'-m','pip','install','-q','--no-cache-dir',PADDLE],check=True)
subprocess.run([VENV_PY,'-m','pip','install','-q','--no-cache-dir','paddlex==3.7.2','paramiko>=3.5,<4'],check=True)
print(subprocess.check_output([VENV_PY,'-c','import sys; print(sys.version)'],text=True).strip(),flush=True)
print('ENV READY',flush=True)


In [ ]:
import subprocess
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=True)
probe=r'''
import os
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK']='True'
import paddle
print('Paddle',paddle.__version__,'CUDA',paddle.device.is_compiled_with_cuda(),flush=True)
if not paddle.device.is_compiled_with_cuda(): raise RuntimeError('Paddle CUDA inactive')
paddle.set_device('gpu:0'); print('DEVICE',paddle.device.get_device(),flush=True)
from paddlex import create_model
print('Downloading/warming PP-DocLayout_plus-L...',flush=True)
m=create_model('PP-DocLayout_plus-L',device='gpu:0')
print('MODEL READY',flush=True)
del m
'''
subprocess.run([VENV_PY,'-u','-c',probe],check=True)


In [ ]:
from google.colab import files
import base64
uploaded=files.upload()
if not uploaded: raise RuntimeError('Upload tml_colab_ed25519')
name,data=next(iter(uploaded.items()))
if name.endswith('.pub'): raise RuntimeError('Upload private key, not .pub')
KEY_B64=base64.b64encode(data).decode()
print('KEY READY',name,flush=True)


In [ ]:
import subprocess
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'; VPS_USER='andre'; VPS_PORT=2222
CLAIM='/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/00_MANIFEST/colab_layout_active_claim.tsv'
STOP='/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/00_MANIFEST/colab_layout_quality_complete_1903.flag'
subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','--short','HEAD'],text=True).strip(); print('CODE',commit,flush=True)
cmd=[VENV_PY,'-u',f'{REPO}/colab/layout_pool.py','--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-key-b64',KEY_B64,'--vps-port',str(VPS_PORT),'--claim',CLAIM,'--stop-flag',STOP,'--workers','4','--downloaders','8','--poll','10']
print('STARTING A100 LAYOUT workers=4 downloaders=8',flush=True)
subprocess.run(cmd,check=True)
